In [3]:
# importing required libraries
import os
from dotenv import load_dotenv
from scripts.scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI
import requests

In [4]:
# loading the environment variables
load_dotenv(override=True) # load the environment variables from the .env file
openai_api_key = os.getenv('OPENAI_API_KEY')
gemini_api_key = os.getenv('GEMINI_API_KEY')
claude_api_key = os.getenv('CLAUDE_API_KEY')


# validate whether the API keys are loaded correctly
# OpenAI
if not openai_api_key:
    print("OPENAI_API_KEY is missing")
elif not openai_api_key.startswith("sk-proj-"):
    print("OPENAI_API_KEY is set, but does not start with sk-proj-")
else:
    print(f"OPENAI_API_KEY loaded, starts with {openai_api_key[:8]}")

# Gemini (this course also uses GOOGLE_API_KEY)
if not gemini_api_key:
    print("GEMINI_API_KEY is missing — check the name in .env")
elif not gemini_api_key.startswith(("AIz", "AQ.")):
    print("GEMINI_API_KEY is set, but does not start with AIz or AQ.")
else:
    print(f"GEMINI_API_KEY loaded, starts with {gemini_api_key[:4]}")

# Claude
if not claude_api_key:
    print("CLAUDE_API_KEY is missing — check the name in .env")
elif not claude_api_key.startswith("sk-ant-"):
    print("CLAUDE_API_KEY is set, but does not start with sk-ant-")
else:
    print(f"CLAUDE_API_KEY loaded, starts with {claude_api_key[:8]}")

OPENAI_API_KEY loaded, starts with sk-proj-
GEMINI_API_KEY loaded, starts with AIza
CLAUDE_API_KEY loaded, starts with sk-ant-a


## **Request Endpoint**

In [5]:
request_endpoint_url = 'https://generativelanguage.googleapis.com/v1beta/openai/chat/completions'

headers = {
    "Authorization": f"Bearer {gemini_api_key}",
    "Content-Type": "application/json"
}

payload = {
    "model": "gemini-3.1-flash-lite",
    "messages": [
        {
            "role": "user",
            "content": "Tell me a fun fact"
        }
    ]
}

In [9]:
response = requests.post( request_endpoint_url, headers=headers, json=payload )
data =response.json()  # This will return the response from the Gemini API
data

{'choices': [{'finish_reason': 'stop',
   'index': 0,
   'message': {'content': 'Here is a fun one for you: **Sea otters hold hands when they sleep.**\n\nThey do this to keep from drifting apart in the water while they snooze. They often form "rafts" by holding onto each other, and they sometimes wrap themselves in giant kelp to stay anchored in one spot!',
    'extra_content': {'google': {'thought_signature': 'EnEKbwERTTIPJS/OnlTJgMOpE9uH9XR5BQWCiGFzSSrGZUV5KQlHSLCBddkQEeCfWo+r9dlQ+sCvt6Wp+m8Cmnn/2zuXFU3IZDCrm7Tc0i2szifYVgT1qQwvSjLh6tfXbLEpGCpSGcU4IEyuU+yBoCpRHg=='}},
    'role': 'assistant'}}],
 'created': 1788381863,
 'id': 'pIqYaoztDM2C-8YPjafXmQc',
 'model': 'gemini-3.1-flash-lite',
 'object': 'chat.completion',
 'usage': {'completion_tokens': 62, 'prompt_tokens': 6, 'total_tokens': 68}}

In [15]:
message = data['choices'][0]['message']['content']
usage = data.get('usage', {})

In [16]:
print("Fun fact:", message)

Fun fact: Here is a fun one for you: **Sea otters hold hands when they sleep.**

They do this to keep from drifting apart in the water while they snooze. They often form "rafts" by holding onto each other, and they sometimes wrap themselves in giant kelp to stay anchored in one spot!


In [14]:
print("Usage:", usage)

Usage: {'completion_tokens': 62, 'prompt_tokens': 6, 'total_tokens': 68}


## **OpenAI Compatible Endpoints**

In [11]:
class PromptBuilder:
    """Fetched page: url + text. Builds chat messages for summarization."""

    SYSTEM_PROMPT = "summarize the content of a website"
    USER_PROMPT_PREFIX = "provide me highlights of the news in bullet points from the content of website:"
    
    def __init__(self, url: str):
        self.url = url
        self.text = fetch_website_contents(url)
        
    def messages(self) -> list[dict]:
        return [
            {"role": "system", "content": self.SYSTEM_PROMPT},
            {"role": "user", "content": self.USER_PROMPT_PREFIX + self.text},
        ]

In [12]:
class ChatModel:
    """Any OpenAI-compatible endpoint (Gemini, Ollama, OpenAI)."""

    def __init__(self, name:str, client: OpenAI, model: str):
        self.name = name
        self.client = client
        self.model = model
        
    def complete(self, messages: list[dict]) -> str:
        # send the messages to the model and return the response
        response = self.client.chat.completions.create(
            model=self.model,
            messages=messages,
        )
        return response.choices[0].message.content

In [13]:
class WebsiteSummarizer:
    """Orchestrates: fetch URL → messages → model → markdown."""

    def __init__(self, chat_model: ChatModel):
        self.chat_model = chat_model

    def summarize(self, url: str) -> str:
        promptBuilder = PromptBuilder(url)
        payload = promptBuilder.messages() # build the messages for the model
        return self.chat_model.complete(payload) # return the model's response

    def display(self, url: str) -> None:
        print(f"{self.chat_model.name} ({self.chat_model.model})\n{url}\n")
        display(Markdown(self.summarize(url))) # display the model's response in markdown format

## **Gemini Chat Completions API**

In [14]:
gemini_model = ChatModel(
    name = 'Gemini',
    client = OpenAI(
        base_url = "https://generativelanguage.googleapis.com/v1beta/openai/",
        api_key = gemini_api_key,
    ),
    model="gemini-3.1-flash-lite",
)

gemini_summarizer = WebsiteSummarizer(gemini_model)
gemini_summarizer.display("https://www.cnn.com/2026/08/30/world/live-news/nepal-china-flood")


Gemini (gemini-3.1-flash-lite)
https://www.cnn.com/2026/08/30/world/live-news/nepal-china-flood



Based on the title provided in your request, here are the highlights regarding the news event:

*   **Rescue Operation Underway:** Emergency responders are actively searching for missing hydropower workers following a disaster.
*   **Cause of the Crisis:** The search efforts were triggered by severe flooding that affected the border region between Nepal and China.
*   **Date of Event:** The reports were dated August 30, 2026.

*(Note: The provided text mostly consisted of website navigation menus, ad feedback forms, and site infrastructure, rather than the body of the news article itself. The summary above is based strictly on the headline provided.)*

## **OLLAMA Chat Completions API**

In [15]:
# test ollama server is up and running
import requests

OLLAMA_URL = "http://localhost:11434/"
print(requests.get(OLLAMA_URL).content)

b'Ollama is running'


https://ollama.com/library/ for exploring different models available in ollama portal

In [16]:
# pull the model
import ollama
ollama.pull("llama3.2:1b")

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [17]:
!ollama list

NAME               ID              SIZE      MODIFIED               
llama3.2:1b        baf6a787fdff    1.3 GB    Less than a second ago    
llama3.2:latest    a80c4f17acd5    2.0 GB    7 hours ago               


In [18]:
llama_model = ChatModel(
    name="Ollama",
    client=OpenAI(base_url = 'http://localhost:11434/' + "v1"),
    model="llama3.2:1b",
)

llama_summarizer = WebsiteSummarizer(llama_model)
llama_summarizer.display("https://www.bbc.com/news")

Ollama (llama3.2:1b)
https://www.bbc.com/news



Here are the highlights of the news in bullet points:

**International**

* Iran has retaliated against the US by launching attacks on US military bases in Iraq and Jordan, according to local state media (BBC News)
* The US has struck back with missile attacks on Iranian military targets (AP News)
* Iranian President Ali Khamenei has warned US that any future attacks will be responded to with "fierce and mighty" retaliation (CNN)
* A group of Iranian prisoners has expressed support for the Saudi Arabian government in the war against Yemen (Fox News)

**US**

* A 23-year-old man in Utah has pleaded not guilty to aggravated murder and six other counts after killing four people at his family's home, according to the judge (Bleeding Gums Murphy's website)
* The US is launching a investigation into the 1995 assassination of The Civil Rights leader Reverend Fred Shuttlesworth (National Public Radio)

**Middle East**

* The US and Iran are at odds over the dispute in Beirut (AP News)
* An earthquake struck Turkey, causing widespread damage and killing over 100 people (Xinhua News Agency)

**World**

* A group of armed separatists in India demanded the Indian government's recognition of a disputed region (BBC News)
* The United Nations warned that the situation in Yemen is escalating, with Iranian aircraft attacking Saudi Arabian targets (UN News)
* A group of researchers has developed a new method for treating cancer that targets healthy cells, while sparing the immune system (Physicist Magazine)

Fun fact demo

In [19]:
display(Markdown(llama_model.complete([{"role": "user", "content": "Tell me a fun fact"}])))

Here's a fun fact: Did you know that there is a type of jellyfish that is immortal? The Turritopsis dohrnii, also known as the "immortal jellyfish," is a species of jellyfish that can transform its body into a younger state through a process called transdifferentiation. This means that it can essentially revert back to its polyp stage and grow back into an adult again, making it theoretically immortal.